# Senario Generation (2025 Baseline vs Renewable Upside)

This notebook generates three senarios from 2025 actual data:

1. baseline: uses 2025 NET_LOAD actuals
2. senario_A: 1.5x solar, 1.0x wind
3. senario_B: 2.0x solar, 1.1x wind

Assumption: ERCOT.LOAD stays at 2025 actual values; only renewable generation is scaled.

In [8]:
from pathlib import Path
import pandas as pd

ROOT = Path('C:/Renewable-PowerGrid-Risk')
SENARIO_DIR = ROOT / 'data' / 'senarios'
input_path = ROOT / 'data' / 'processed' / 'hourly_load_renewable_merged.csv'

df = pd.read_csv(input_path, parse_dates=['datetime'])
print(f'Loaded: {input_path}')
print(df.shape)

Loaded: C:\Renewable-PowerGrid-Risk\data\processed\hourly_load_renewable_merged.csv
(26310, 23)


In [9]:
required_cols = ['datetime', 'ERCOT.LOAD', 'ERCOT.PVGR.GEN', 'ERCOT.WIND.GEN', 'NET_LOAD']
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f'Missing required columns: {missing}')

df_2025 = df[df['datetime'].dt.year == 2025].copy()
if df_2025.empty:
    raise ValueError('No 2025 rows found in source data.')

print('2025 rows:', len(df_2025))
df_2025.head()

2025 rows: 8762


,datetime,Date,ERCOT.LOAD,ERCOT.WIND.GEN,"Total Wind Installed, MW","Wind Output, % of Load","Wind Output, % of Installed",Wind 1-hr MW change,Wind 1-hr % change,ERCOT.PVGR.GEN,...,Solar 1-hr MW change,Solar 1-hr % change,Daytime Hour,Ramping Daytime Hour,hour,month,renewable,renewable_capacity,renewable_share,NET_LOAD
17547,2025-01-01 00:00:00,2025-01-01,44715.455903,13285.939850,39357,29.712187,33.757501,1481.889068,12.554072,0.703511,...,0.0,0.0,NaN,NaN,0,1,13286.643361,68150,0.297138,31428.812542
17548,2025-01-01 01:00:00,2025-01-01,44110.291011,14369.247057,39357,32.575725,36.510016,NaN,NaN,0.724839,...,NaN,NaN,False,False,1,1,14369.971896,68150,0.325774,29740.319115
17549,2025-01-01 02:00:00,2025-01-01,43795.576070,14888.718368,39357,33.995941,37.829912,519.471311,3.615160,0.743425,...,0.0,0.0,False,False,2,1,14889.461793,68150,0.339976,28906.114277
17550,2025-01-01 03:00:00,2025-01-01,43630.220156,15306.719255,39357,35.082838,38.891987,418.000887,2.807501,0.742429,...,0.0,0.0,False,False,3,1,15307.461684,68150,0.350845,28322.758472
17551,2025-01-01 04:00:00,2025-01-01,43519.323668,15025.925240,39357,34.527019,38.178533,-280.794015,-1.834449,0.752879,...,0.0,0.0,False,False,4,1,15026.678119,68150,0.345287,28492.645548


In [10]:
# Baseline: use actual 2025 NET_LOAD
baseline = df_2025[['datetime', 'ERCOT.LOAD', 'ERCOT.PVGR.GEN', 'ERCOT.WIND.GEN', 'NET_LOAD']].copy()
baseline['scenario'] = 'baseline'

# Senario A: 1.5x solar, 1.0x wind
senario_a = df_2025[['datetime', 'ERCOT.LOAD', 'ERCOT.PVGR.GEN', 'ERCOT.WIND.GEN']].copy()
senario_a['ERCOT.PVGR.GEN'] = senario_a['ERCOT.PVGR.GEN'] * 1.5
senario_a['ERCOT.WIND.GEN'] = senario_a['ERCOT.WIND.GEN'] * 1.1
senario_a['NET_LOAD'] = senario_a['ERCOT.LOAD'] - senario_a['ERCOT.PVGR.GEN'] - senario_a['ERCOT.WIND.GEN']
senario_a['scenario'] = 'senario_A'

# Senario B: 2.0x solar, 1.1x wind
senario_b = df_2025[['datetime', 'ERCOT.LOAD', 'ERCOT.PVGR.GEN', 'ERCOT.WIND.GEN']].copy()
senario_b['ERCOT.PVGR.GEN'] = senario_b['ERCOT.PVGR.GEN'] * 2.0
senario_b['ERCOT.WIND.GEN'] = senario_b['ERCOT.WIND.GEN'] * 1.2
senario_b['NET_LOAD'] = senario_b['ERCOT.LOAD'] - senario_b['ERCOT.PVGR.GEN'] - senario_b['ERCOT.WIND.GEN']
senario_b['scenario'] = 'senario_B'

scenario_all = pd.concat([baseline, senario_a, senario_b], ignore_index=True)
scenario_all = scenario_all.sort_values(['scenario', 'datetime']).reset_index(drop=True)

print(scenario_all['scenario'].value_counts())
scenario_all.head()

scenario
baseline     8762
senario_A    8762
senario_B    8762
Name: count, dtype: int64


,datetime,ERCOT.LOAD,ERCOT.PVGR.GEN,ERCOT.WIND.GEN,NET_LOAD,scenario
0,2025-01-01 00:00:00,44715.455903,0.703511,13285.939850,31428.812542,baseline
1,2025-01-01 01:00:00,44110.291011,0.724839,14369.247057,29740.319115,baseline
2,2025-01-01 02:00:00,43795.576070,0.743425,14888.718368,28906.114277,baseline
3,2025-01-01 03:00:00,43630.220156,0.742429,15306.719255,28322.758472,baseline
4,2025-01-01 04:00:00,43519.323668,0.752879,15025.925240,28492.645548,baseline


In [11]:
summary = scenario_all.groupby('scenario')[['ERCOT.LOAD', 'ERCOT.PVGR.GEN', 'ERCOT.WIND.GEN', 'NET_LOAD']].agg(['mean', 'max', 'min'])
summary.round(2)

ERCOT.LOAD                    ERCOT.PVGR.GEN                  \
                mean       max      min           mean       max   min   
scenario                                                                 
baseline    56436.17  83876.46  38617.9        7705.52  29503.09  0.02   
senario_A   56436.17  83876.46  38617.9       11558.28  44254.63  0.03   
senario_B   56436.17  83876.46  38617.9       15411.04  59006.17  0.04   

          ERCOT.WIND.GEN                    NET_LOAD                      
                    mean       max     min      mean       max       min  
scenario                                                                  
baseline        13119.36  28264.50  168.65  35611.29  69401.91  11492.38  
senario_A       14431.30  31090.95  185.52  30446.59  68080.15    -57.81  
senario_B       15743.23  33917.40  202.38  25281.90  67695.42 -13783.46

In [12]:
out_all = SENARIO_DIR / 'senario_generation_2025_all.csv'
out_base = SENARIO_DIR / 'senario_generation_2025_baseline.csv'
out_a = SENARIO_DIR / 'senario_generation_2025_senario_A.csv'
out_b = SENARIO_DIR / 'senario_generation_2025_senario_B.csv'

scenario_all.to_csv(out_all, index=False)
baseline.to_csv(out_base, index=False)
senario_a.to_csv(out_a, index=False)
senario_b.to_csv(out_b, index=False)

print('Saved files:')
print(out_all)
print(out_base)
print(out_a)
print(out_b)

Saved files:
C:\Renewable-PowerGrid-Risk\data\senarios\senario_generation_2025_all.csv
C:\Renewable-PowerGrid-Risk\data\senarios\senario_generation_2025_baseline.csv
C:\Renewable-PowerGrid-Risk\data\senarios\senario_generation_2025_senario_A.csv
C:\Renewable-PowerGrid-Risk\data\senarios\senario_generation_2025_senario_B.csv
